In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMRegressor
import gradio as gr

In [13]:
data = pd.read_csv("./DATA/insurance_data.csv")
data.head()

,index,PatientID,age,gender,bmi,bloodpressure,diabetic,children,smoker,region,claim
0,0,1,39.0,male,23.2,91,Yes,0,No,southeast,1121.87
1,1,2,24.0,male,30.1,87,No,0,No,southeast,1131.51
2,2,3,NaN,male,33.3,82,Yes,0,No,southeast,1135.94
3,3,4,NaN,male,33.7,80,No,0,No,northwest,1136.40
4,4,5,NaN,male,34.1,100,No,0,No,northwest,1137.01


In [14]:
data.dropna(axis=0,inplace=True)
data.head()

,index,PatientID,age,gender,bmi,bloodpressure,diabetic,children,smoker,region,claim
0,0,1,39.0,male,23.2,91,Yes,0,No,southeast,1121.87
1,1,2,24.0,male,30.1,87,No,0,No,southeast,1131.51
7,7,8,19.0,male,41.1,100,No,0,No,northwest,1146.80
8,8,9,20.0,male,43.0,86,No,0,No,northwest,1149.40
9,9,10,30.0,male,53.1,97,No,0,No,northwest,1163.46


In [15]:
features = ['age','gender','bmi','bloodpressure','diabetic','children','smoker','region']
target = 'claim'
categorical_cols = ['gender','diabetic','smoker','region']
label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  data[col] = le.fit_transform(data[col])
  label_encoders[col] = le
X = data[features]
y = data[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LGBMRegressor(random_state=42)
model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000655 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 342
[LightGBM] [Info] Number of data points in the train set: 1065, number of used features: 8
[LightGBM] [Info] Start training from score 13337.716900


LGBMRegressor(random_state=42)

In [20]:
def predict_claims(csv_file):
  input_data = pd.read_csv(csv_file.name)
  patient_ids = input_data['PatientID']
  for col in categorical_cols:
    le = label_encoders[col]
    input_data[col] = input_data[col].map(lambda x: le.transform([x])[0] if x in le.classes_ else 0)
  X_input = input_data[features]
  predictions = model.predict(X_input)
  results = pd.DataFrame({
    'PatientID': patient_ids,
    'predictedClaims': predictions
  })
  return results

In [ ]:
iface = gr.Interface(
  fn = predict_claims,
  inputs = gr.File(file_types=[".csv"]),
  outputs = "dataframe",
  title = "Insurance Claim Prediction (LightGBM)",
  description = "Upload CSV file with PatientID, age, gender, bmi, blood pressure, diabetic status, amount of children, smoking status, region"
)
iface.launch()

* Running on local URL:  http://127.0.0.1:7862

To create a public link, set `share=True` in `launch()`.


Created dataset file at: .gradio\flagged\dataset1.csv
